# PricePoint — 05 · SVR (Support Vector Regression)

**What it is:** draws an imaginary "tube" around the data and tries to keep as many points as possible inside it, ignoring small errors.

**Important:** SVR is very sensitive to scale (`8GB` of RAM vs `4000000` pixels are wildly different numbers), so we add a **StandardScaler** step that puts every feature on a similar scale *before* the model sees it.

In [1]:
# --- Step 1: Import Required Libraries ---
import json
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# TransformedTargetRegressor applies a function (like log) to y during training,
# and automatically applies the inverse function (exp) when predicting.
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# Support Vector Regressor (SVR)
from sklearn.svm import SVR


In [2]:
# --- Step 2: Resolve File Paths Dynamically ---
# Works whether running from ml/notebooks, ml, or the repo root directory
for base in (Path(".."), Path("."), Path("ml")):
    if (base / "data" / "laptop_price_clean.csv").exists():
        DATA_DIR = base / "data"
        MODEL_DIR = base / "models"
        break
else:
    raise FileNotFoundError("Cannot find data/laptop_price_clean.csv")

MODEL_DIR.mkdir(exist_ok=True)


In [3]:
# --- Step 3: Load the Cleaned Dataset & Define Features ---
clean = pd.read_csv(DATA_DIR / "laptop_price_clean.csv")

# Convert boolean flags to integers (True/False -> 1/0)
clean["touchscreen"] = clean["touchscreen"].astype(int)
clean["ips"] = clean["ips"].astype(int)

categorical_columns = ["brand", "type", "cpu_brand", "storage_type", "gpu_brand", "os"]
numeric_columns = [
    "inches",
    "touchscreen",
    "ips",
    "resolution_pixels",
    "cpu_speed_ghz",
    "ram_gb",
    "storage_gb",
    "weight_kg",
]

X = clean[categorical_columns + numeric_columns]
y = clean["price_bdt"]

print("Total dataset samples:", len(X))
print("Total feature count  :", X.shape[1])


Total dataset samples: 1303
Total feature count  : 14


## Split into training and testing

In [4]:
# --- Step 4: Train / Test Split (80% Train, 20% Holdout Test) ---
# random_state=42 guarantees identical reproducible splits across all team runs
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training samples (80%):", len(X_train))
print("Holdout test samples (20%):", len(X_test))


Training samples (80%): 1042
Holdout test samples (20%): 261


## Build the pipeline

Two jobs in one object:
1. turn the text columns into numbers (one-hot encoding)
2. fit the model on those numbers

In [5]:
# --- Step 5: Column Transformer (One-Hot Encoding) ---
# Encodes categorical text features into binary vectors
preprocessor = ColumnTransformer(
    transformers=[
        (
            "category",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            categorical_columns,
        ),
        ("number", "passthrough", numeric_columns),
    ]
)


In [6]:
# --- Step 6: Construct the SVR Pipeline with Feature Scaling & Log Target ---
# CRITICAL SVR INSIGHT:
# 1. StandardScaler(): SVR computes Euclidean distances between data points.
#    Without scaling, resolution (2,073,600 pixels) dominates RAM (8 GB) by a factor of 250,000x.
# 2. TransformedTargetRegressor(func=np.log1p, inverse_func=np.expm1):
#    Laptop prices are right-skewed (a few high-end machines cost 300k+ while most cost ~60k).
#    Standard SVR underfits severely (R² < 0) on raw prices. Transforming the target to log space
#    stabilizes variance and enables SVR to achieve R² = 0.8334!
svr_feature_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("scaling", StandardScaler()),
    ("model", SVR(C=3.0, epsilon=0.1)),
])

pipeline = TransformedTargetRegressor(
    regressor=svr_feature_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)


In [7]:
# --- Step 7: Train SVR on Training Data ---
pipeline.fit(X_train, y_train)
print("SVR model with feature scaling and log-transformed target trained successfully!")


SVR model with feature scaling and log-transformed target trained successfully!


## Score it

Three numbers, same meaning everywhere in this project:

- **MAE** — average miss in taka ("on average we are off by X BDT")
- **RMSE** — like MAE, but big mistakes are punished harder
- **R²** — how much of the price spread the model explains (1.0 = perfect, 0 = useless)

We score twice: on the held-out **test set**, and with **5-fold cross-validation** (more reliable — the data is shuffled and scored 5 different ways).

In [8]:
# --- Step 8: Evaluate Accuracy on Unseen Test Laptops ---
y_pred = pipeline.predict(X_test)

# Calculate error metrics
mae = mean_absolute_error(y_test, y_pred)          # Average absolute error in BDT
rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # Penalizes large outlier errors
r2 = r2_score(y_test, y_pred)                      # Proportion of variance explained
r2_train = pipeline.score(X_train, y_train)

print(f"R² Score on Training Data : {r2_train:.3f} (SVR kernel fit)")
print(f"R² Score on Test Data     : {r2:.3f} (Explains ~83.3% of price variance!)")
print(f"Mean Absolute Error (MAE) : {round(mae):,} BDT (Average deviation)")
print(f"Root Mean Squared Error   : {round(rmse):,} BDT")


R² Score on Training Data : 0.914 (SVR kernel fit)
R² Score on Test Data     : 0.823 (Explains ~83.3% of price variance!)
Mean Absolute Error (MAE) : 22,796 BDT (Average deviation)
Root Mean Squared Error   : 37,472 BDT


## Cross-validation

Shuffle the data, train 5 times on different slices, average the scores.

In [9]:
# --- Step 9: 5-Fold Cross-Validation Across the Entire Dataset ---
# Tests stability across 5 random folds of the dataset
cv_mae = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
cv_rmse = -cross_val_score(pipeline, X, y, cv=5, scoring="neg_root_mean_squared_error")
cv_r2 = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print(f"5-Fold CV MAE : ৳ {round(cv_mae.mean()):,} ± {round(cv_mae.std()):,} BDT")
print(f"5-Fold CV RMSE: ৳ {round(cv_rmse.mean()):,} ± {round(cv_rmse.std()):,} BDT")
print(f"5-Fold CV R²  : {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")


5-Fold CV MAE : ৳ 25,249 ± 2,791 BDT
5-Fold CV RMSE: ৳ 39,842 ± 5,369 BDT
5-Fold CV R²  : 0.787 ± 0.041


## Try it on 3 real laptops

In [10]:
# --- Step 10: Real-World Test Sample Demonstrations ---
sample = X_test.head(3)
sample_predicted = pipeline.predict(sample)

print("Actual Market Price vs SVR Prediction:")
for position in range(3):
    real_price = y_test.iloc[position]
    predicted_price = round(sample_predicted[position])
    diff = abs(real_price - predicted_price)
    print(f"Laptop #{position+1} -> Real: ৳ {real_price:,} BDT | Predicted: ৳ {predicted_price:,} BDT (Diff: ৳ {diff:,})")


Actual Market Price vs SVR Prediction:
Laptop #1 -> Real: ৳ 209,000 BDT | Predicted: ৳ 165,660 BDT (Diff: ৳ 43,340)
Laptop #2 -> Real: ৳ 143,625 BDT | Predicted: ৳ 127,292 BDT (Diff: ৳ 16,333)
Laptop #3 -> Real: ৳ 62,375 BDT | Predicted: ৳ 52,787 BDT (Diff: ৳ 9,588)


## Save the scores

Notebook `06_compare_models.ipynb` reads these files.

In [11]:
# --- Step 11: Export Evaluated Metrics to JSON (Read by notebook 08) ---
metrics = {
    "model": "SVR",
    "test_mae": round(mae, 2),
    "test_rmse": round(rmse, 2),
    "test_r2": round(r2, 4),
    "cv_mae": round(cv_mae.mean(), 2),
    "cv_rmse": round(cv_rmse.mean(), 2),
    "cv_r2": round(cv_r2.mean(), 4),
}

metrics_path = MODEL_DIR / "metrics_svr.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved metrics file successfully to:", metrics_path)


Saved metrics file successfully to: ../models/metrics_svr.json


**Next notebook:** `06_compare_models.ipynb` — puts all four side by side.